# 09 — Measure exact semantic similarity

## Control-plane contract

- **Purpose:** M8 frozen BGE-M3 vectors → exact cosine matrices → article body aggregation → top-k semantic relations.
- **Inputs:** M8 title/body/entity vector+metadata artifacts, M7 canonical place/event relations, versioned M9 config.
- **Outputs:** run-scoped M9 matrices, recipe evidence, directed article/article and article/entity relations, QA and manifests.
- **Dependencies:** `M8_COMMIT=b8e01c3`, `BGE-M3@5617a9f…`, NumPy/Pandas only.
- **Parameters:** title/body weights selected from the three predeclared recipes; body max/top-3 weights `.70/.30`; top-k `20/5`; `EXACT_COSINE`; `float32`; seed `42`.
- **Assumptions:** vectors are normalized; M7 relations are weak evaluation labels, never score features.
- **Side effects:** writes only `data/40_semantic/mbn/m9/run_20260808_m9_similarity/`.
- **Fail conditions:** M8 contract drift, invalid vectors/FKs, rank/PK violations, formula mismatch, or manifest mismatch.


In [1]:
from pathlib import Path
import sys
REPO_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO_ROOT))
from src.text.m9_similarity import run_m9, output_root
m9_gate = run_m9(REPO_ROOT, force_matrices=True)
m9_gate


{'m9Gate': 'M9_SEMANTIC_READY',
 'modelContract': {'m8CommitPinned': True,
  'modelId': True,
  'modelRevision': True,
  'shapes': True,
  'float32': True,
  'metaRows': True,
  'modelMetadata': True,
  'nanInfFree': True,
  'verdict': 'M9_MODEL_CONTRACT_PASS'},
 'qualityChecks': {'selfRelationCount': 0,
  'articleRelationDuplicatePk': 0,
  'entityRelationDuplicatePk': 0,
  'brokenArticleFK': 0,
  'brokenEntityFK': 0,
  'nanCount': 0,
  'infCount': 0,
  'rankViolationCount': 0,
  'topKViolationCount': 0},
 'formulaMismatchCount': 0,
 'manifestHashMismatch': 0,
 'canonicalNotebookExpected': 'notebooks/09MeasureSemanticSimilarity.ipynb'}

In [2]:
import json, pandas as pd
OUT = output_root(REPO_ROOT)
article = pd.read_parquet(OUT / 'article_article_semantic_relation.parquet')
entity = pd.read_parquet(OUT / 'article_entity_semantic_relation.parquet')
quality = json.loads((OUT / 'semantic_relation_quality_report.json').read_text())
print('ROW COUNTS')
print({'Articles': quality['counts']['articles'], 'Chunks': quality['counts']['chunks'], 'Entities': quality['counts']['entities'], 'ArticleArticleRelations': len(article), 'ArticleEntityRelations': len(entity)})
print('NULL / DUPLICATE / SELF / FK')
print({'nulls': int(article.isna().sum().sum()+entity.isna().sum().sum()), 'duplicates': quality['checks']['articleRelationDuplicatePk']+quality['checks']['entityRelationDuplicatePk'], 'selfRelation': quality['checks']['selfRelationCount'], 'brokenFK': quality['checks']['brokenArticleFK']+quality['checks']['brokenEntityFK'], 'formulaMismatch': 0})
print('QUALITY METRICS')
display(pd.read_parquet(OUT / 'similarity_recipe_benchmark.parquet'))
print('PERFORMANCE', quality['performance'])
print('OUTPUT PATH', OUT)
print('NEXT: 10ClassifyCultureContent.ipynb')


ROW COUNTS
{'Articles': 461, 'Chunks': 1062, 'Entities': 32, 'ArticleArticleRelations': 9220, 'ArticleEntityRelations': 2305}
NULL / DUPLICATE / SELF / FK
{'nulls': 0, 'duplicates': 0, 'selfRelation': 0, 'brokenFK': 0, 'formulaMismatch': 0}
QUALITY METRICS


,recipeVersion,recipeName,titleWeight,bodyWeight,evaluationAuthority,articleEntity_queryCount,articleEntity_recallAt1,articleEntity_recallAt3,articleEntity_recallAt5,articleEntity_recallAt10,articleEntity_mrr,sharedGeo_queryCount,sharedGeo_recallAt5,sharedGeo_recallAt10,sharedGeo_recallAt20,sharedGeo_mrr
0,m9_title_only@1,TITLE_ONLY,1.000000,0.000000,DIRECT_GEO_RELATION_SILVER + M8_INTERNAL_RETRI...,25,0.44,0.68,0.72,0.88,0.590350,3,0.333333,0.333333,0.666667,0.186111
1,m9_body_only@1,BODY_ONLY,0.000000,1.000000,DIRECT_GEO_RELATION_SILVER + M8_INTERNAL_RETRI...,25,0.84,0.96,1.00,1.00,0.896667,3,0.000000,0.000000,0.000000,0.008866
2,m9_hybrid_baseline@1,HYBRID_BASELINE,0.466667,0.533333,DIRECT_GEO_RELATION_SILVER + M8_INTERNAL_RETRI...,25,0.84,0.92,0.96,1.00,0.890000,3,0.000000,0.000000,0.333333,0.029930


PERFORMANCE {'loadNormalizeSeconds': 0.0071404320042347535, 'titleMatrixSeconds': 0.02310652500455035, 'chunkMatrixSeconds': 0.03168584399827523, 'entityMatricesSeconds': 0.008066606998909265, 'bodyAggregationSeconds': 1.3309643210013746, 'recipeBenchmarkSeconds': 0.026413526000396814, 'topKMaterializationSeconds': 1.0078452560046571, 'totalSeconds': 3.5990191730015795, 'peakRAMMB': 238.703125}
OUTPUT PATH /home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE/data/40_semantic/mbn/m9/run_20260808_m9_similarity
NEXT: 10ClassifyCultureContent.ipynb
